In [46]:
"""
Cocokkan data sample (output format_dataset.py = list of string "role: message")
dengan raw MultiWOZ 2.2 yang mengandung slot-value.

Kunci penting (disesuaikan dengan multiwoz_to_qiscus_format.py + format_dataset.py):
  - room_id  == dialogue_id MultiWOZ
  - role     == "user" jika speaker=user, selain itu "agent"
  - tiap turn di-render sebagai  f"{role}: {utterance}"
  - satu dialog = "\n".join(semua turn)   <-- string kanonik untuk hashing

Cara pakai:
    main(inspect=True)
    main(input_file='train',
         sample_path='all_dialogue_test_500.json',
         out_path='matched.json')
"""

import os
import re
import glob
import json
import hashlib
import pathlib

import pandas as pd


# ----------------------------------------------------------------------
# 1. RENDER KANONIK — HARUS identik dengan pipeline Anda
# ----------------------------------------------------------------------
def speaker_to_role(speaker: str) -> str:
    """Sama persis dengan multiwoz_to_qiscus_format.py."""
    return "user" if str(speaker).lower() == "user" else "agent"


def normalize_line(line: str) -> str:
    """Normalisasi ringan per-baris untuk menyerap beda spasi/kapital.
    Dipakai SETELAH render 'role: message' agar adil di kedua sisi."""
    return re.sub(r"\s+", " ", line.strip().lower())


def render_dialog_from_turns(turns) -> list:
    """Raw -> list baris 'role: utterance' (meniru format_dataset.py)."""
    lines = []
    for t in turns:
        role = speaker_to_role(t.get("speaker", ""))
        utt = t.get("utterance", t.get("text", ""))
        lines.append(f"{role}: {utt}")
    return lines


def lines_to_key(lines) -> str:
    """Gabung baris 'role: message' -> string kanonik -> hash."""
    canonical = "\n".join(normalize_line(l) for l in lines)
    return hashlib.sha1(canonical.encode("utf-8")).hexdigest()


# ----------------------------------------------------------------------
# 2. INSPEKSI
# ----------------------------------------------------------------------
def inspect_raw(raw_dir):
    files = sorted(glob.glob(os.path.join(raw_dir, "*.json")))
    print(f"Ditemukan {len(files)} file JSON di {raw_dir}\n")
    if not files:
        return
    data = json.load(open(files[0], "r", encoding="utf-8"))
    print(f"--- {os.path.basename(files[0])} ---  root: {type(data).__name__}")
    dlist = data if isinstance(data, list) else list(data.values())
    if dlist:
        d0 = dlist[0]
        print("Keys per dialog:", list(d0.keys()))
        turns = d0.get("turns", [])
        if turns:
            print("Keys per turn:", list(turns[0].keys()))
            print("Contoh turn:",
                  json.dumps(turns[0], ensure_ascii=False)[:800])


# ----------------------------------------------------------------------
# 3. LOAD RAW + slot
# ----------------------------------------------------------------------
def extract_slots_from_turn(turn) -> dict:
    slots = {}
    for frame in turn.get("frames", []):
        sv = frame.get("state", {}).get("slot_values", {})
        for slot, values in sv.items():
            slots.setdefault(slot, set()).update(
                values if isinstance(values, list) else [values]
            )
    return slots


def load_raw(raw_dir):
    files = sorted(glob.glob(os.path.join(raw_dir, "*.json")))
    dialogues = []
    for fp in files:
        data = json.load(open(fp, "r", encoding="utf-8"))
        dlist = data if isinstance(data, list) else list(data.values())
        for d in dlist:
            did = d.get("dialogue_id")
            turns = d.get("turns", [])
            lines = render_dialog_from_turns(turns)
            merged = {}
            for t in turns:
                for slot, vals in extract_slots_from_turn(t).items():
                    merged.setdefault(slot, set()).update(vals)
            dialogues.append({
                "dialogue_id": did,
                "key": lines_to_key(lines),
                "slots": {k: sorted(v) for k, v in merged.items()},
            })
    print(f"[raw] {len(dialogues)} dialog dari {len(files)} file")
    return dialogues


# ----------------------------------------------------------------------
# 4. LOAD SAMPLE (list of string, tiap string = dialog ber-\n)
# ----------------------------------------------------------------------
def load_sample(sample_path):
    # with open(sample_path, "r", encoding="utf-8") as f:
    #     data = json.load(f)

    data = pd.read_json(sample_path, orient='records', lines=True)

    data = data["dialogue"].to_list()
    
    # data = data.to_json(orient="records", lines=True).splitlines()

    # format_dataset.py menghasilkan list of string
    if not isinstance(data, list):
        raise ValueError("Format sample tak terduga: harusnya list of string.")
    print(data[0])
    samples = []
    for idx, item in enumerate(data):
        if isinstance(item, str):
            lines = item.split("\n")            # pecah balik ke baris turn
        elif isinstance(item, list):            # antisipasi list of list
            lines = item
        else:
            lines = [str(item)]
        samples.append({
            "sample_idx": idx,
            "key": lines_to_key(lines),
        })
    print(f"[sample] {len(samples)} dialog dimuat")
    return samples


# ----------------------------------------------------------------------
# 5. MATCH
# ----------------------------------------------------------------------
def match_exact(raw, samples):
    index = {}
    for r in raw:
        index.setdefault(r["key"], []).append(r)
    mapping, unmatched = [], []
    for s in samples:
        hit = index.get(s["key"])
        if hit:
            r = hit[0]
            mapping.append({"sample_idx": s["sample_idx"],
                            "dialogue_id": r["dialogue_id"],
                            "slots": r["slots"], "match": "exact"})
        else:
            unmatched.append(s)
    return mapping, unmatched


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main(inspect=False,
         input_file='train',
         sample_path='all_dialogue_test_500.jsonl',
         out_path='matched.json'):

    base_dir = pathlib.Path().resolve().parent
    raw_dir = os.path.join(base_dir, f"multiwoz/data/MultiWOZ_2.2/{input_file}")

    if inspect:
        inspect_raw(raw_dir)
        return

    raw = load_raw(raw_dir)
    samples = load_sample(sample_path)

    mapping, unmatched = match_exact(raw, samples)
    rate = len(mapping) / max(len(samples), 1)
    print(f"[match] exact: {len(mapping)}/{len(samples)} ({rate:.1%}), "
          f"gagal: {len(unmatched)}")

    json.dump(
        {"matched": mapping,
         "failed": [{"sample_idx": s["sample_idx"]} for s in unmatched]},
        open(out_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"[done] -> {out_path}")

    if unmatched:
        print(f"[warn] {len(unmatched)} gagal exact-match. Jika banyak, "
              f"berarti teks raw vs sample berbeda (cek normalisasi). "
              f"Catatan: room_id = dialogue_id, jadi idealnya bisa "
              f"join via room_id tanpa teks — lihat catatan di chat.")


if __name__ == "__main__":
    main()

[raw] 8437 dialog dari 17 file
user: I want to find some information about a hotel called alexander bed and breakfast.
agent: Alexander bed and breakfast is a guesthouse in the cheap price range in the centre of town.
user: Do they have reservations available for 6 people and 3 nights starting from Thursday?
agent: Booking was unsuccessful. Would you like to find another hotel.
[sample] 499 dialog dimuat
[match] exact: 499/499 (100.0%), gagal: 0
[done] -> matched.json
